# Layer 1: CHSH & Mermin Inequalities

**Goal**: Prove entanglement via Bell-type inequalities on 2–6 qubits.

| Witness | State | Classical bound | Quantum maximum |
|---------|-------|----------------|----------------|
| CHSH | Singlet |`\|S\| ≤ 2` | `2√2 ≈ 2.83` |
| Mermin-n | GHZ-n | `2^(n/2)` | `2^(n-1)` |

Violation of these inequalities **proves** entanglement — no classical hidden-variable theory can reproduce the results.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.backend import get_backend

# --- Set your token here or via env variable IQM_TOKEN ---
import os
TOKEN = os.environ.get('IQM_TOKEN', None)  # or: TOKEN = 'paste-token-here'
DEVICE = 'emerald'  # or 'garnet'
USE_HARDWARE = TOKEN is not None

backend = get_backend(token=TOKEN, device=DEVICE)
print(f'Backend: {backend}')
print(f'Running on hardware: {USE_HARDWARE}')

## 1. CHSH Inequality

Prepare the singlet state $|\psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$ and measure:
$$S = \langle A_1 B_1 \rangle - \langle A_1 B_2 \rangle + \langle A_2 B_1 \rangle + \langle A_2 B_2 \rangle$$

Classical bound: $|S| \leq 2$. Quantum maximum: $|S| = 2\sqrt{2} \approx 2.828$.

In [ ]:
from src.witnesses.chsh import run_chsh, build_chsh_circuits

SHOTS = 4000
result_chsh = run_chsh(backend, shots=SHOTS)

print(f"S = {result_chsh['S']:.4f}")
print(f"Classical bound: |S| ≤ {result_chsh['classical_bound']}")
print(f"Quantum ideal:   |S| = {abs(result_chsh['S_ideal']):.4f}")
print(f"Violation:  {result_chsh['violation']:.4f}  ({result_chsh['significance_sigma']:.1f}σ)")
print(f"\nIndividual correlators:")
from src.witnesses.chsh import corr_from_counts
for key, counts in result_chsh['counts'].items():
    print(f"  {key}: {corr_from_counts(counts):.4f}")

In [ ]:
# Visualise CHSH measurement circuits
circuits = build_chsh_circuits()
print('Circuit for A1B1 (singlet + measurement rotations):')
circuits['A1B1'].draw('mpl', style='clifford')

## 2. Mermin Inequalities (n = 3, 4, 5)

The Mermin operator $M_n = \text{Re}\left((X+iY)^{\otimes n}\right)$ sums all $n$-qubit Pauli products with an **even** number of $Y$ operators:

$$M_n = \sum_{k=0,2,4,\ldots} (-1)^{k/2} \sum_{|S|=k} \bigotimes_{i\in S} Y_i \otimes \bigotimes_{j\notin S} X_j$$

| $n$ | Classical bound $2^{n/2}$ | Quantum max $2^{n-1}$ | Violation ratio |
|-----|--------------------------|----------------------|----------------|
| 3 | 2.83 | 4 | 1.41 |
| 4 | 4.00 | 8 | 2.00 |
| 5 | 5.66 | 16 | 2.83 |

In [ ]:
from src.witnesses.mermin import run_mermin, classical_bound, quantum_maximum, num_circuits
from src.circuits.ghz import build_ghz_no_measure

mermin_results = {}
for n in [3, 4, 5]:
    print(f'\n--- Mermin-{n} ({num_circuits(n)} measurement circuits) ---')
    ghz = build_ghz_no_measure(n)
    res = run_mermin(backend, n, ghz, shots=SHOTS)
    mermin_results[n] = res
    print(f'  M_{n} = {res["M_n"]:.4f}')
    print(f'  Classical bound: {res["classical_bound"]:.4f}')
    print(f'  Quantum maximum: {res["quantum_maximum"]:.4f}')
    print(f'  Violation: {res["violation"]:.4f}  (ratio: {res["violation_ratio"]:.3f}x)')

In [ ]:
# Plot: Mermin violation ratio vs n
ns = list(mermin_results.keys())
ratios_measured = [mermin_results[n]['violation_ratio'] for n in ns]
ratios_ideal = [quantum_maximum(n) / classical_bound(n) for n in ns]
bounds = [classical_bound(n) for n in ns]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(ns, ratios_measured, color='#32a8a4', alpha=0.8, label='Measured')
ax.plot(ns, ratios_ideal, 'r--o', label='Ideal (quantum)')
ax.axhline(1.0, color='gray', linestyle=':', label='Classical bound (ratio=1)')
ax.set_xlabel('Number of qubits n')
ax.set_ylabel('|M_n| / classical bound')
ax.set_title('Mermin Violation Ratio')
ax.legend()
ax.set_xticks(ns)

ax = axes[1]
ax.bar(ns, [abs(mermin_results[n]['M_n']) for n in ns],
       color='#e84545', alpha=0.8, label='|M_n| measured')
ax.plot(ns, [quantum_maximum(n) for n in ns], 'b--o', label='Quantum max')
ax.plot(ns, bounds, 'g:s', label='Classical bound')
ax.set_xlabel('Number of qubits n')
ax.set_ylabel('|M_n|')
ax.set_title('Mermin Operator Values')
ax.legend()
ax.set_xticks(ns)

plt.suptitle('Mermin Inequality Violations on IQM Hardware', fontsize=13)
plt.tight_layout()
plt.savefig('mermin_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved mermin_results.png')

## Summary

Both CHSH and Mermin inequalities demonstrate that **no local hidden-variable theory** can reproduce our hardware results — entanglement must be present.

In [ ]:
print('=== LAYER 1 SUMMARY ===')
print(f'CHSH: S = {result_chsh["S"]:.3f}, violation = {result_chsh["violation"]:.3f} ({result_chsh["significance_sigma"]:.1f}σ)')
for n, res in mermin_results.items():
    status = 'VIOLATES' if res['violation'] > 0 else 'no violation'
    print(f'Mermin-{n}: M = {res["M_n"]:.3f}, bound = {res["classical_bound"]:.3f} -> {status} (ratio {res["violation_ratio"]:.2f}x)')